# Kenya Hockey — International Performance (Data Collection)

This notebook scrapes Kenya's national-team results from continental and
international competitions organized by the African Hockey Federation
(AfHF) and the International Hockey Federation (FIH) — the Africa Cup
of Nations, Junior Africa Cup, and related events — as a companion to
the domestic KHU pipeline (`01_scraping.ipynb` onward).

**Data source, confirmed by direct verification, not assumed:** results
for these competitions are recorded on FIH's Tournament Management
System (`tms.fih.ch`), not on the African Hockey Federation's own
website. A real completed match page was checked directly before
writing any scraper code, and the data available there is genuinely
richer than the domestic KHU data in one specific way: it distinguishes
**Field Goal vs Penalty Corner vs Penalty Stroke** for every goal —
exactly the breakdown confirmed absent from the domestic data source
(see Module 36 in `03_advanced_analytics.ipynb`). It also includes full
lineups, venue (with coordinates), weather at pushback, and match
officials.

**Scope, stated honestly:** this notebook covers *national-team*
competitions only (Africa Cup of Nations, Junior Africa Cup, and
similar FIH-sanctioned representative events). The **Africa Cup for
Club Champions** — where Kenyan clubs (confirmed: Sikh Union Nairobi
won Bronze in a recent edition) compete — was investigated but not
confirmed to live on this same platform; it appears to be reported
primarily on africahockey.org's own site instead, which would need
its own separate scraping reconnaissance, the same way KHU's site did
originally. That is a documented next step, not part of this notebook.

**Competitions covered so far (add more by extending `COMPETITIONS`
below — confirm each new entry's competition ID directly before
adding it, the same way these were confirmed):**
- Africa Cup of Nations (Men) 2017 — competition 801
- Africa Cup of Nations (Men) 2022 — competition 1384
- Africa Cup of Nations (Women) 2022 — competition 1385
- Africa Cup of Nations (Men) 2025 — competition 1815
- Africa Cup of Nations (Women) 2025 — competition 1814
- Junior Africa Cup (Women) 2024 — competition 1771

**On historical depth:** this spans 2017–2025, not just the current
season — but it is not a complete history. FIH's own system states
reliable data only goes back to 2013 (2012-and-earlier is still being
digitised by FIH), and several editions between 2013–2025 haven't
been added yet (each one needs its own competition ID confirmed the
same way, not guessed). Extending this list is the easiest way to add
value to this notebook going forward.

**Requirements:** Chrome + ChromeDriver (managed automatically via
`webdriver-manager`), `selenium`, `beautifulsoup4`, `pandas`.


## Setup

In [1]:
# Cell 0 - Install Dependencies Not in the Standard Anaconda/Jupyter Setup

# pdfplumber isn\'t part of a typical Python or Anaconda install, unlike
# pandas/selenium/etc — confirmed by a real run hitting
# "ModuleNotFoundError: No module named \'pdfplumber\'" on a fresh
# environment. Running this cell first makes the notebook self-contained
# rather than relying on a separate manual pip install step, which is
# easy to miss.
#
# Uses subprocess directly (plain Python) rather than a Jupyter "!pip
# install" magic command — the two behave the same inside a notebook,
# but plain Python is real, testable code, not notebook-only syntax.
# Safe to re-run: pip simply reports "already satisfied" if a package
# is already installed.

import subprocess
import sys

def ensure_installed(package):
    try:
        __import__(package)
        print(f"{package} already installed.")
    except ImportError:
        print(f"Installing {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])
        except subprocess.CalledProcessError:
            print(f"\n\u26a0 Automatic install failed for {package}. This can happen in some "
                  f"restricted Python environments. Try running this in a terminal instead:\n"
                  f"    pip install {package}\n"
                  f"or, if you see an \'externally managed environment\' error specifically:\n"
                  f"    pip install --break-system-packages {package}")
            raise

ensure_installed("pdfplumber")
ensure_installed("requests")

print("\nDependencies ready.")


pdfplumber already installed.
requests already installed.

Dependencies ready.


In [2]:
# Cell 1 - Imports

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time
import io

print("Imports ready.")


Imports ready.


In [3]:
# Cell 2 - Configure Chrome

HEADLESS = True

chrome_options = Options()

if HEADLESS:
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1920,1080")

chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)


def create_driver():
    d = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    d.execute_script("""
    Object.defineProperty(navigator, 'webdriver', {
        get: () => undefined
    })
    """)
    return d


driver = create_driver()

print(f"Chrome launched successfully. (headless={HEADLESS})")


Chrome launched successfully. (headless=True)


## Competitions

Curated, confirmed list — each competition ID below was checked
directly against the live site before being added here.


In [4]:
# Cell 3 - Confirmed Kenya-Relevant Competitions

COMPETITIONS = {
    "AFCON-M-2017": {
        "id": 801,
        "name": "Africa Cup of Nations (Men) 2017",
        "url": "https://tms.fih.ch/competitions/801",
    },
    "AFCON-M-2022": {
        "id": 1384,
        "name": "Africa Cup of Nations (Men) 2022",
        "url": "https://tms.fih.ch/competitions/1384",
    },
    "AFCON-W-2022": {
        "id": 1385,
        "name": "Africa Cup of Nations (Women) 2022",
        "url": "https://tms.fih.ch/competitions/1385",
    },
    "AFCON-M-2025": {
        "id": 1815,
        "name": "Africa Cup of Nations (Men) 2025",
        "url": "https://tms.fih.ch/competitions/1815",
    },
    "AFCON-W-2025": {
        "id": 1814,
        "name": "Africa Cup of Nations (Women) 2025",
        "url": "https://tms.fih.ch/competitions/1814",
    },
    "JAC-W-2024": {
        "id": 1771,
        "name": "Junior Africa Cup (Women) 2024",
        "url": "https://tms.fih.ch/competitions/1771",
    },
    # Confirmed to exist but not yet added (add more editions the same way:
    # find a real team page for that edition, confirm the competition ID
    # in its breadcrumb, then add an entry here):
    #   - 2013, 2015, 2019, 2021, 2023 Africa Cup of Nations editions
    #   - 2017 Women\'s Africa Cup of Nations
    #   - Other Junior Africa Cup / Hockey5s editions Kenya has played in
    # Reliable data only goes back to 2013 — FIH\'s own system states
    # 2012-and-earlier is still being digitised, so that\'s a hard floor,
    # not a gap in this scraper.
}

KENYA_CODE = "KEN"

print(f"Tracking {len(COMPETITIONS)} confirmed competitions.")


Tracking 6 confirmed competitions.


## Discover Kenya's Matches in Each Competition

In [5]:
# Cell 4 - Discover Kenya Match Links (and Tournament Standings)

# Each competition summary page lists every match as a pair of team-code
# links (e.g. "NAM - KEN") pointing to /matches/{id}. This finds every
# match link on the page, then keeps only the ones where Kenya's 3-letter
# code appears in the surrounding text — catching both "KEN - X" and
# "X - KEN" fixtures.
#
# It also captures the tournament\'s final standings table from the same
# page load — no extra page fetch needed, since it\'s already sitting on
# the competition summary page. This gives Kenya\'s results real context
# (e.g. "lost to Egypt, who won every match" vs "lost to Egypt, who barely
# beat anyone") without paying the cost of scraping full match detail for
# every team in the tournament, which isn\'t needed for a Kenya-focused
# project. Only the standings are collected tournament-wide; goals, cards,
# and lineups (the expensive per-match scrape) stay Kenya-only, below.

kenya_match_links = {}   # competition_key -> list of match URLs
all_standings = []       # one row per team per competition

MAX_RETRIES_INTL_DISCOVERY = 3

for comp_key, comp in COMPETITIONS.items():

    print(f"Opening {comp_key}: {comp['url']}")

    # Confirmed real gap, fixed: this cell previously had no retry
    # protection at all — a single transient network blip
    # (net::ERR_NAME_NOT_RESOLVED, confirmed directly from a real run)
    # crashed the entire cell immediately, unlike every other scraping
    # loop in this project, which recovers from exactly this kind of
    # blip by retrying and recreating the driver if needed.
    page_loaded = False
    for attempt in range(1, MAX_RETRIES_INTL_DISCOVERY + 1):
        try:
            driver.get(comp["url"])
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            page_loaded = True
            break
        except TimeoutException:
            print(f"  ⚠ Attempt {attempt}/{MAX_RETRIES_INTL_DISCOVERY}: page did not load in time for {comp_key}.")
            time.sleep(2)
        except Exception as e:
            err = str(e)
            session_dead = (
                "ERR_NAME_NOT_RESOLVED" in err
                or "invalid session id" in err
                or "session deleted" in err
                or "chrome not reachable" in err
                or "Read timed out" in err
                or "Connection refused" in err
            )
            if session_dead:
                print(f"  ⚠ Attempt {attempt}/{MAX_RETRIES_INTL_DISCOVERY}: network/browser issue "
                      f"({err.splitlines()[0][:80]}). Restarting driver...")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = create_driver()
                time.sleep(2)
            else:
                print(f"  ⚠ Attempt {attempt}/{MAX_RETRIES_INTL_DISCOVERY} failed: {err.splitlines()[0][:80]}")
                time.sleep(2)

    if not page_loaded:
        print(f"  ✗ Giving up on {comp_key} after {MAX_RETRIES_INTL_DISCOVERY} attempts — "
              f"check your internet connection, or re-run this cell later.")
        kenya_match_links[comp_key] = []
        continue

    time.sleep(2)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # --- Match discovery (unchanged) ---
    match_links = []
    for a in soup.find_all("a", href=True):
        if "/matches/" not in a["href"]:
            continue
        context = a.get_text(" ", strip=True)
        parent_text = a.find_parent().get_text(" ", strip=True) if a.find_parent() else ""
        if KENYA_CODE in context or KENYA_CODE in parent_text:
            full_url = a["href"] if a["href"].startswith("http") else f"https://tms.fih.ch{a['href']}"
            match_links.append(full_url)

    match_links = sorted(set(match_links))

    if not match_links:
        print(f"  ⚠ No Kenya matches found on this page — check the competition ID and URL are correct.")
    else:
        print(f"  ✓ Found {len(match_links)} Kenya match(es)")

    kenya_match_links[comp_key] = match_links

    # --- Standings extraction ---
    # First attempt: check column names directly (works if the table has
    # a clean single-row header). Second attempt, if that finds nothing:
    # scan the first few ROWS for identifying text instead, the same fix
    # that was needed for the ACCC medal table below, which turned out to
    # use a merged multi-row header that pandas doesn\'t parse as columns
    # at all. This competition-standings case hasn\'t been directly
    # diagnosed against a real page the way the match-detail and ACCC
    # fixes were, so this is a reasoned defensive fix, not a confirmed
    # one — if it still doesn\'t work, that\'s the next thing to diagnose.
    try:
        tables = pd.read_html(io.StringIO(html), flavor="lxml")
    except (ValueError, ImportError):
        tables = []

    found_standings = 0

    for t in tables:
        cols = [str(c).strip().lower() for c in t.columns]
        has_team = "team" in cols
        has_points = any(c in cols for c in ["pts", "points"])
        has_played = any(c in cols for c in ["p", "played", "pld"])

        if has_team and has_points and has_played:
            t = t.copy()
            t["CompetitionKey"] = comp_key
            all_standings.append(t)
            found_standings += 1
            continue

        # Fallback: scan first 3 rows for standings-like header text
        header_row_idx = None
        for row_idx in range(min(3, len(t))):
            row_text = " ".join(str(v).strip().lower() for v in t.iloc[row_idx].tolist())
            if "team" in row_text and ("pts" in row_text or "points" in row_text):
                header_row_idx = row_idx
                break

        if header_row_idx is not None:
            header_values = [str(v).strip() for v in t.iloc[header_row_idx].tolist()]
            data_rows = t.iloc[header_row_idx + 1:].copy()
            data_rows.columns = header_values[:len(data_rows.columns)]
            data_rows["CompetitionKey"] = comp_key
            all_standings.append(data_rows)
            found_standings += 1

    if found_standings == 0:
        print(f"  ⚠ No standings table found for {comp_key} — tournament context "
              f"will be unavailable for this competition, but Kenya\'s own match "
              f"results are unaffected.")
    else:
        print(f"  ✓ Captured {found_standings} standings table(s)")

total_found = sum(len(v) for v in kenya_match_links.values())
print(f"\nTotal Kenya matches discovered: {total_found}")
print(f"Total standings tables captured: {len(all_standings)}")


Opening AFCON-M-2017: https://tms.fih.ch/competitions/801


  ✓ Found 5 Kenya match(es)
  ⚠ No standings table found for AFCON-M-2017 — tournament context will be unavailable for this competition, but Kenya's own match results are unaffected.
Opening AFCON-M-2022: https://tms.fih.ch/competitions/1384


  ✓ Found 4 Kenya match(es)
  ⚠ No standings table found for AFCON-M-2022 — tournament context will be unavailable for this competition, but Kenya's own match results are unaffected.
Opening AFCON-W-2022: https://tms.fih.ch/competitions/1385


  ✓ Found 5 Kenya match(es)
  ⚠ No standings table found for AFCON-W-2022 — tournament context will be unavailable for this competition, but Kenya's own match results are unaffected.
Opening AFCON-M-2025: https://tms.fih.ch/competitions/1815


  ✓ Found 6 Kenya match(es)
  ⚠ No standings table found for AFCON-M-2025 — tournament context will be unavailable for this competition, but Kenya's own match results are unaffected.
Opening AFCON-W-2025: https://tms.fih.ch/competitions/1814


  ✓ Found 5 Kenya match(es)
  ⚠ No standings table found for AFCON-W-2025 — tournament context will be unavailable for this competition, but Kenya's own match results are unaffected.
Opening JAC-W-2024: https://tms.fih.ch/competitions/1771


  ✓ Found 5 Kenya match(es)
  ⚠ No standings table found for JAC-W-2024 — tournament context will be unavailable for this competition, but Kenya's own match results are unaffected.

Total Kenya matches discovered: 30
Total standings tables captured: 0


## Match Scraper

In [6]:
# Cell 5 - International Match Scraper Function

# Uses pandas.read_html() to extract the Goals and Cards tables by
# matching on their actual column headers, rather than guessing CSS
# class names sight-unseen — more robust to the page's exact styling,
# since these are genuine HTML <table> elements on this site (confirmed
# by direct inspection before writing this function).

def scrape_international_match(match_url, competition_key):

    driver.get(match_url)

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        raise RuntimeError(f"Page did not load in time: {match_url}")

    time.sleep(2)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text("\n", strip=True)

    # --- Teams and score ---
    # The page title area reads like "Kenya" / "1 - 4" / "Zimbabwe" as
    # separate headings. Fall back gracefully if this exact shape isn\'t
    # found — never silently invent a score.
    score_match = re.search(r"\b(\d{1,2})\s*-\s*(\d{1,2})\b(?!\s*-)", page_text)

    # Confirmed via direct inspection of a real page: headings appear in
    # this order: [Competition Name, Round Label ("RR"/"Pool A"/etc.),
    # HomeTeam, Score, AwayTeam, "Goals", "Card Detail", ...]. The
    # original filter only excluded headings starting with a digit, which
    # correctly dropped the competition name and score, but NOT the round
    # label ("RR", "Pool A") — those aren\'t digits, so they were wrongly
    # picked up as team_candidates[0], silently corrupting HomeTeam for
    # every match and breaking every downstream "is this Kenya\'s match?"
    # check even though the underlying goal/card data was fine.
    ROUND_LABEL_PATTERN = re.compile(
        r"^(One Pool|Pool [A-Z0-9]+|RR|Semi-final|Final|3rd\s*-?\s*4th.*?|5th.*?|Quarter-final)$",
        re.IGNORECASE
    )

    headings = [h.get_text(strip=True) for h in soup.find_all(["h1", "h2", "h3"]) if h.get_text(strip=True)]
    team_candidates_from_headings = [
        h for h in headings
        if not re.match(r"^\d", h)
        and len(h) > 1
        and not ROUND_LABEL_PATTERN.match(h)
        and "Cup" not in h  # excludes competition-name headings that don\'t start with
        and "-" not in h    # a digit on some page template variants (e.g. "Africa Cup of
                            # Nations (M) - 2025"), confirmed present on real 2025-edition pages
    ]

    home_team = team_candidates_from_headings[0] if len(team_candidates_from_headings) > 0 else None
    away_team = team_candidates_from_headings[1] if len(team_candidates_from_headings) > 1 else None

    home_score, away_score = (None, None)
    if score_match:
        home_score, away_score = int(score_match.group(1)), int(score_match.group(2))

    # --- Competition / round ---
    # The round/stage label appears near the top of the page, right after
    # the competition name and before the team names — searching the
    # WHOLE page risks matching an unrelated mention of "Pool A" etc.
    # further down (e.g. inside the Match Details section), so this is
    # scoped to roughly the first 400 characters of page text only.
    round_search_zone = page_text[:400]
    round_match = re.search(
        r"(One Pool|Pool [A-Z0-9]|RR|Semi-final|Final|3rd\s*-?\s*4th.*?|5th.*?|Quarter-final)",
        round_search_zone
    )
    round_name = round_match.group(0) if round_match else None

    # --- Date / Venue (from the Match Details table) ---
    match_date, venue = None, None
    date_match = re.search(r"(\d{4}-\d{2}-\d{2})\s+\d{2}:\d{2}", page_text)
    if date_match:
        match_date = date_match.group(1)

    venue_match = re.search(r"Venue\s+([A-Za-z0-9 ,()\-]+?)(?:\s+https|\n)", page_text)
    if venue_match:
        venue = venue_match.group(1).strip()

    # --- Goals table ---
    # flavor="lxml" explicitly, so pandas doesn\'t attempt an html5lib
    # fallback (not installed in every environment) when a page happens
    # to have no tables at all — e.g. a scoreless match with no goals
    # or cards to report, the same situation confirmed for many domestic
    # KHU matches.
    try:
        tables = pd.read_html(io.StringIO(html), flavor="lxml")
    except (ValueError, ImportError):
        tables = []

    goals = []
    for t in tables:
        cols = [str(c).strip().lower() for c in t.columns]
        if "action" in cols and "player" in cols and "minute" in cols:
            for _, row in t.iterrows():
                goals.append({
                    "team": row.get("Team"),
                    "minute": row.get("Minute"),
                    "player": row.get("Player"),
                    "action": row.get("Action"),
                    "score_after": row.get("Score"),
                })
            break

    # --- Cards table ---
    cards = []
    for t in tables:
        cols = [str(c).strip().lower() for c in t.columns]
        if "type" in cols and "player" in cols and "minute" in cols and "action" not in cols:
            for _, row in t.iterrows():
                cards.append({
                    "team": row.get("Team"),
                    "minute": row.get("Minute"),
                    "player": row.get("Player"),
                    "type": row.get("Type"),
                })
            break

    # Correction pass: prefer team names confirmed by the Goals/Cards
    # tables\' own "Team" column over the heading-based guess above,
    # since the heading text format is confirmed inconsistent across
    # different competition editions on this site (e.g. some pages show
    # "RR"/"Pool A" or even the competition name itself as a heading in
    # the same position a team name would be, depending on the page
    # template used for that specific year). The events tables\' "Team"
    # values are directly what was recorded against each goal/card, so
    # they don\'t have this ambiguity. Only applies when this yields
    # exactly two distinct teams; falls back to the heading-based guess
    # otherwise (e.g. a scoreless match with no events to confirm from).
    teams_seen_in_order = []
    for e in goals + cards:
        t = e.get("team")
        if t and t not in teams_seen_in_order:
            teams_seen_in_order.append(t)

    if len(teams_seen_in_order) == 2:
        # Use heading order to decide which of the two confirmed teams
        # is "home" — still useful for this one purpose even though the
        # heading list isn\'t reliable for discovering team names outright.
        ordered_by_heading = [h for h in headings if h in teams_seen_in_order]
        if len(ordered_by_heading) == 2:
            home_team, away_team = ordered_by_heading[0], ordered_by_heading[1]
        else:
            home_team, away_team = teams_seen_in_order[0], teams_seen_in_order[1]

    return {
        "competition_key": competition_key,
        "home_team": home_team,
        "away_team": away_team,
        "home_score": home_score,
        "away_score": away_score,
        "round": round_name,
        "date": match_date,
        "venue": venue,
        "goals": goals,
        "cards": cards,
        "url": match_url,
    }


print("scrape_international_match() ready.")


scrape_international_match() ready.


## Batch Scraping

In [7]:
# Cell 6 - Scrape All Kenya Matches

all_intl_matches = []
failed_intl_matches = []

total = sum(len(v) for v in kenya_match_links.values())
done = 0

for comp_key, links in kenya_match_links.items():

    for url in links:

        done += 1
        print(f"[{comp_key}] {done}/{total}")

        try:
            data = scrape_international_match(url, comp_key)
            all_intl_matches.append(data)

            score_txt = (
                f"{data['home_score']} - {data['away_score']}"
                if data["home_score"] is not None else "score unknown"
            )
            print(f"  ✓ {data['home_team']} {score_txt} {data['away_team']} ({len(data['goals'])} goals, {len(data['cards'])} cards recorded)")

        except Exception as e:
            print(f"  ⚠ Failed: {str(e).splitlines()[0]}")
            failed_intl_matches.append({"competition_key": comp_key, "url": url})

print(f"\nFinished. Scraped {len(all_intl_matches)} of {total} Kenya matches.")
if failed_intl_matches:
    print(f"{len(failed_intl_matches)} match(es) failed — re-run this cell to retry, or check the URLs manually:")
    for f in failed_intl_matches:
        print(f"  [{f['competition_key']}] {f['url']}")


[AFCON-M-2017] 1/30


  ✓ Ghana 3 - 0 Kenya (3 goals, 3 cards recorded)
[AFCON-M-2017] 2/30


  ✓ Egypt 4 - 1 Kenya (5 goals, 3 cards recorded)
[AFCON-M-2017] 3/30


  ✓ Kenya 2 - 1 Nigeria (3 goals, 7 cards recorded)
[AFCON-M-2017] 4/30


  ✓ South Africa 6 - 1 Kenya (7 goals, 2 cards recorded)
[AFCON-M-2017] 5/30


...

  ✓ Kenya 0 - 3 Namibia (3 goals, 1 cards recorded)
[JAC-W-2024] 27/30


  ✓ Uganda 0 - 0 Kenya (1 goals, 4 cards recorded)
[JAC-W-2024] 28/30


  ✓ Kenya 0 - 4 South Africa (4 goals, 1 cards recorded)
[JAC-W-2024] 29/30


  ✓ Zambia 1 - 1 Kenya (2 goals, 6 cards recorded)
[JAC-W-2024] 30/30


  ✓ Kenya 1 - 4 Zimbabwe (5 goals, 3 cards recorded)

Finished. Scraped 30 of 30 Kenya matches.


## Build & Save DataFrames

In [8]:
# Cell 7 - Build Matches & Events DataFrames

intl_matches_df = pd.DataFrame([
    {
        "CompetitionKey": m["competition_key"],
        "Competition": COMPETITIONS[m["competition_key"]]["name"],
        "Round": m["round"],
        "Date": m["date"],
        "Venue": m["venue"],
        "HomeTeam": m["home_team"],
        "AwayTeam": m["away_team"],
        "HomeGoals": m["home_score"],
        "AwayGoals": m["away_score"],
        "URL": m["url"],
    }
    for m in all_intl_matches
])

intl_events = []
for m in all_intl_matches:
    for g in m["goals"]:
        intl_events.append({
            "CompetitionKey": m["competition_key"],
            "HomeTeam": m["home_team"],
            "AwayTeam": m["away_team"],
            "Date": m["date"],
            "EventType": "Goal",
            "GoalType": g["action"],   # Field Goal / Penalty Corner / Penalty Stroke
            "Team": g["team"],
            "Player": g["player"],
            "Minute": g["minute"],
        })
    for c in m["cards"]:
        intl_events.append({
            "CompetitionKey": m["competition_key"],
            "HomeTeam": m["home_team"],
            "AwayTeam": m["away_team"],
            "Date": m["date"],
            "EventType": "Card",
            "GoalType": None,
            "Team": c["team"],
            "Player": c["player"],
            "Minute": c["minute"],
            "CardType": c["type"],
        })

intl_events_df = pd.DataFrame(intl_events)

print(f"Matches: {len(intl_matches_df)}")
print(f"Events : {len(intl_events_df)}")

intl_matches_df.head()


Matches: 30
Events : 239


,CompetitionKey,Competition,Round,Date,Venue,HomeTeam,AwayTeam,HomeGoals,AwayGoals,URL
0,AFCON-M-2017,Africa Cup of Nations (Men) 2017,RR,2017-10-22,Pitch 1,Ghana,Kenya,3,0,https://tms.fih.ch/matches/10697
1,AFCON-M-2017,Africa Cup of Nations (Men) 2017,RR,2017-10-23,Pitch 1,Egypt,Kenya,4,1,https://tms.fih.ch/matches/10700
2,AFCON-M-2017,Africa Cup of Nations (Men) 2017,RR,2017-10-25,Pitch 1,Kenya,Nigeria,2,1,https://tms.fih.ch/matches/10702
3,AFCON-M-2017,Africa Cup of Nations (Men) 2017,RR,2017-10-28,Pitch 1,South Africa,Kenya,6,1,https://tms.fih.ch/matches/10708
4,AFCON-M-2017,Africa Cup of Nations (Men) 2017,NaN,2017-10-29,Pitch 1,Ghana,Kenya,5,3,https://tms.fih.ch/matches/10712


In [9]:
# Cell 8 - Save International Data

import os
os.makedirs("../data/raw", exist_ok=True)

intl_matches_df.to_csv("../data/raw/international_matches.csv", index=False)
intl_events_df.to_csv("../data/raw/international_events.csv", index=False)

print("Saved international_matches.csv and international_events.csv to data/raw/.")


Saved international_matches.csv and international_events.csv to data/raw/.


## Kenya's International Record

Basic analytics, deliberately kept simple given the small number of
matches available — a handful of tournament appearances, not a full
domestic season. The goal-type breakdown (Field Goal / Penalty Corner /
Penalty Stroke) is genuinely new information this project didn\'t have
access to domestically.


In [10]:
# Cell 9 - Kenya Results Summary

def kenya_result(row):
    if row["HomeTeam"] == "Kenya":
        gf, ga = row["HomeGoals"], row["AwayGoals"]
        opponent = row["AwayTeam"]
    else:
        gf, ga = row["AwayGoals"], row["HomeGoals"]
        opponent = row["HomeTeam"]
    if pd.isna(gf) or pd.isna(ga):
        outcome = "Unknown"
    elif gf > ga:
        outcome = "Win"
    elif gf < ga:
        outcome = "Loss"
    else:
        outcome = "Draw"
    return pd.Series({"Opponent": opponent, "GF": gf, "GA": ga, "Result": outcome})


kenya_results = intl_matches_df.join(intl_matches_df.apply(kenya_result, axis=1))

summary = (
    kenya_results
    .groupby("Competition")
    .agg(
        Played=("Result", "count"),
        Wins=("Result", lambda x: (x == "Win").sum()),
        Draws=("Result", lambda x: (x == "Draw").sum()),
        Losses=("Result", lambda x: (x == "Loss").sum()),
        GoalsFor=("GF", "sum"),
        GoalsAgainst=("GA", "sum"),
    )
    .reset_index()
)

print("Kenya\'s international record by competition:")
summary


Kenya's international record by competition:


,Competition,Played,Wins,Draws,Losses,GoalsFor,GoalsAgainst
0,Africa Cup of Nations (Men) 2017,5,1,0,4,7,19
1,Africa Cup of Nations (Men) 2022,4,1,0,3,8,12
2,Africa Cup of Nations (Men) 2025,6,2,0,4,10,15
3,Africa Cup of Nations (Women) 2022,5,3,1,1,10,5
4,Africa Cup of Nations (Women) 2025,5,3,1,1,8,5
5,Junior Africa Cup (Women) 2024,5,0,2,3,2,12


In [11]:
# Cell 9B - Consolidate Tournament Standings

if len(all_standings) == 0:
    print("No standings tables were captured — tournament context won\'t be "
          "available, but Kenya\'s own results above are unaffected.")
    standings_df = pd.DataFrame()
else:
    def normalize_standings_table(t):
        # Renamed per-table, BEFORE concatenation — renaming after
        # combining tables with different original header names (e.g.
        # "P"/"Played", "Pts"/"Points") creates duplicate columns once
        # concat unions every distinct column name it has ever seen,
        # which breaks downstream arithmetic. Normalizing first means
        # every table already speaks the same column names when combined.
        rename_map = {}
        for col in t.columns:
            c = str(col).strip().lower()
            if c in ("p", "played", "pld"):
                rename_map[col] = "Played"
            elif c in ("pts", "points"):
                rename_map[col] = "Points"
            elif c in ("gf", "for"):
                rename_map[col] = "GF"
            elif c in ("ga", "against"):
                rename_map[col] = "GA"
            elif c == "team":
                rename_map[col] = "Team"
        t = t.rename(columns=rename_map)
        keep_cols = [c for c in ["CompetitionKey", "Team", "Played", "Points", "GF", "GA"] if c in t.columns]
        return t[keep_cols]

    normalized_tables = [normalize_standings_table(t) for t in all_standings]
    standings_df = pd.concat(normalized_tables, ignore_index=True, sort=False)

    standings_df.to_csv("../data/processed/international_standings.csv", index=False)
    print(f"Captured {len(standings_df)} team-tournament standings rows across "
          f"{standings_df['CompetitionKey'].nunique()} competition(s).")

standings_df.head(10)


No standings tables were captured — tournament context won't be available, but Kenya's own results above are unaffected.


""


In [12]:
# Cell 9C - Kenya vs Tournament Average

if len(standings_df) == 0:
    print("No standings available — skipping this comparison.")
else:
    tournament_avg = (
        standings_df
        .groupby("CompetitionKey")
        .agg(AvgPoints=("Points", "mean"), AvgGF=("GF", "mean"), AvgGA=("GA", "mean"))
        .reset_index()
    )

    kenya_standing = standings_df[standings_df["Team"] == "Kenya"].merge(
        tournament_avg, on="CompetitionKey"
    )

    kenya_standing["PointsVsAverage"] = (kenya_standing["Points"] - kenya_standing["AvgPoints"]).round(1)
    kenya_standing["GFVsAverage"] = (kenya_standing["GF"] - kenya_standing["AvgGF"]).round(1)

    print("Kenya\'s performance relative to each tournament\'s average team:")
    print("(Positive PointsVsAverage/GFVsAverage = above tournament average)")

    kenya_standing[["CompetitionKey", "Points", "AvgPoints", "PointsVsAverage", "GF", "AvgGF", "GFVsAverage"]]


No standings available — skipping this comparison.


In [13]:
# Cell 9D - Men vs Women Performance Comparison

# Gender is derived from each competition\'s own name, which this
# notebook already writes consistently as "... (Men) ..." / "... (Women) ..."
# in the COMPETITIONS dict above — not an assumption about the data
# itself, just reading back what was already defined.

def infer_gender(competition_name):
    return "Women" if "Women" in str(competition_name) else "Men"

kenya_results["Gender"] = kenya_results["Competition"].apply(infer_gender)

gender_comparison = (
    kenya_results
    .groupby("Gender")
    .agg(
        Played=("Result", "count"),
        Wins=("Result", lambda x: (x == "Win").sum()),
        Draws=("Result", lambda x: (x == "Draw").sum()),
        Losses=("Result", lambda x: (x == "Loss").sum()),
        GoalsFor=("GF", "sum"),
        GoalsAgainst=("GA", "sum"),
    )
    .reset_index()
)

gender_comparison["WinRate"] = (gender_comparison["Wins"] / gender_comparison["Played"] * 100).round(1)
gender_comparison["DrawRate"] = (gender_comparison["Draws"] / gender_comparison["Played"] * 100).round(1)
gender_comparison["LossRate"] = (gender_comparison["Losses"] / gender_comparison["Played"] * 100).round(1)

print("Kenya\'s international record, Men vs Women:")
print(gender_comparison)

gender_comparison.to_csv("../data/processed/international_gender_comparison.csv", index=False)
print("\nSaved international_gender_comparison.csv")

gender_comparison


Kenya's international record, Men vs Women:
  Gender  Played  Wins  Draws  Losses  GoalsFor  GoalsAgainst  WinRate  \
0    Men      15     4      0      11        25            46     26.7   
1  Women      15     6      4       5        20            22     40.0   

   DrawRate  LossRate  
0       0.0      73.3  
1      26.7      33.3  

Saved international_gender_comparison.csv


,Gender,Played,Wins,Draws,Losses,GoalsFor,GoalsAgainst,WinRate,DrawRate,LossRate
0,Men,15,4,0,11,25,46,26.7,0.0,73.3
1,Women,15,6,4,5,20,22,40.0,26.7,33.3


In [14]:
# Cell 10B - Kenya International Top Scorers (by name)

# The Goal Type Breakdown above aggregates by FG/PC/PS category, not by
# player — this is the actual "who scored" table, which was missing
# entirely from both the notebook and dashboard until now.

kenya_scorers = intl_events_df[
    (intl_events_df["EventType"] == "Goal") &
    (intl_events_df["Team"] == "Kenya")
]

if len(kenya_scorers) > 0:
    international_top_scorers = (
        kenya_scorers
        .groupby("Player")
        .size()
        .reset_index(name="Goals")
        .sort_values("Goals", ascending=False)
    )
    print(f"{len(international_top_scorers)} different Kenya player(s) with a recorded international goal:")
    print(international_top_scorers)

    international_top_scorers.to_csv("../data/processed/international_top_scorers.csv", index=False)
    print("\nSaved international_top_scorers.csv")
else:
    international_top_scorers = pd.DataFrame(columns=["Player", "Goals"])
    print("No Kenya goals recorded yet — run the batch scraping cell above first.")


24 different Kenya player(s) with a recorded international goal:
              Player  Goals
23  WAKHURA Constant      6
2        BWIRE Grace      3
19    ONYANGO Festus      3
3     CHEBET Eleanor      2
20    OPONDO Aurelia      2
10   MASAMBU Bethuel      2
5       KEMUNTO Naom      2
4       ININGU Allan      2
18    OMWAKA Mathias      2
21       OWITI Alice      2
1     AWINO Emmanuel      1
0         ANJAO Joan      1
7        LOISE Moses      1
6      KIPYEGO Titus      1
9      MANGO Kennedy      1
8       LUDIALI Ivan      1
15  NAMACHANJA Daina      1
14   MWAGANDI Brenda      1
13     MUTRIA George      1
12     MUTIVA Flavia      1
11      MOSOIKO Alex      1
17      OMARI Cliffe      1
16    NAYOMBE Linton      1
22   USAGI Sutcliffe      1

Saved international_top_scorers.csv


In [15]:
# Cell 10 - Goal Type Breakdown (Field Goal / Penalty Corner / Penalty Stroke)

# This distinction is confirmed absent from the domestic KHU data
# (see Module 36, 03_advanced_analytics.ipynb) but is fully present here
# — a genuinely new analytical capability this data source provides.

kenya_goals = intl_events_df[
    (intl_events_df["EventType"] == "Goal") &
    (intl_events_df["Team"] == "Kenya")
]

if len(kenya_goals) > 0:
    goal_type_breakdown = (
        kenya_goals
        .groupby("GoalType")
        .size()
        .reset_index(name="Count")
        .sort_values("Count", ascending=False)
    )
    print("Kenya\'s goals by type, across all recorded international matches:")
    print(goal_type_breakdown)

    goal_type_breakdown.to_csv("../data/processed/international_goal_types.csv", index=False)
    print("\nSaved international_goal_types.csv")
else:
    goal_type_breakdown = pd.DataFrame(columns=["GoalType", "Count"])
    print("No Kenya goals recorded yet in the scraped matches — "
          "run the batch scraping cell above first.")


Kenya's goals by type, across all recorded international matches:
         GoalType  Count
0      Field Goal     23
1  Penalty Corner     16
2  Penalty Stroke      1

Saved international_goal_types.csv


In [16]:
# Cell 11 - Kenya Match Log

kenya_results_sorted = kenya_results.sort_values("Date", na_position="last")

kenya_results_sorted[
    ["Competition", "Round", "Date", "Venue", "Opponent", "GF", "GA", "Result"]
]


,Competition,Round,Date,Venue,Opponent,GF,GA,Result
0,Africa Cup of Nations (Men) 2017,RR,2017-10-22,Pitch 1,Ghana,0,3,Loss
1,Africa Cup of Nations (Men) 2017,RR,2017-10-23,Pitch 1,Egypt,1,4,Loss
2,Africa Cup of Nations (Men) 2017,RR,2017-10-25,Pitch 1,Nigeria,2,1,Win
3,Africa Cup of Nations (Men) 2017,RR,2017-10-28,Pitch 1,South Africa,1,6,Loss
4,Africa Cup of Nations (Men) 2017,NaN,2017-10-29,Pitch 1,Ghana,3,5,Loss
5,Africa Cup of Nations (Men) 2022,Pool A,2022-01-18,Weather at Pushback,Namibia,4,1,Win
10,Africa Cup of Nations (Women) 2022,Pool B,2022-01-18,Weather at Pushback,No goals scored,5,0,Win
6,Africa Cup of Nations (Men) 2022,Pool A,2022-01-19,Weather at Pushback,South Africa,1,2,Loss
9,Africa Cup of Nations (Women) 2022,Pool B,2022-01-19,Weather at Pushback,Zambia,3,0,Win
11,Africa Cup of Nations (Women) 2022,Pool B,2022-01-20,Weather at Pushback,Nigeria,2,1,Win


In [17]:
# Cell 12 - Save Analytics Outputs

summary.to_csv("../data/processed/international_summary.csv", index=False)
kenya_results_sorted.to_csv("../data/processed/international_match_log.csv", index=False)

print("International analytics saved to data/processed/.")


International analytics saved to data/processed/.


## Domestic → International Player Linkage

Checks whether any player who scored a goal or was carded in the domestic
KHU league also appears in Kenya\'s international squads — a genuine,
citable connection ("this domestic player earned international caps")
that neither dataset can show on its own.

**Name-matching problem, solved explicitly rather than assumed away:**
the two sources use different name formats. Domestic events use
`"Joan Ajao"` (first name first). International records use
`"AJAO Joan"` (surname first, capitalized). A plain string comparison
would silently find zero matches even for the same real person. The fix
below normalizes both to an unordered set of lowercase name tokens, so
format differences don\'t block a real match.

**Important limitation, stated plainly:** this is a **name match, not a
confirmed identity match**. Two different people can share a name.
Every result below should be treated as "worth checking," not as
verified fact — confirm manually (e.g. does the timeline make sense?
does the player\'s club affiliation line up?) before citing a specific
match anywhere.


In [18]:
# Cell 13 - Link Domestic and International Players

import os

domestic_events_path = "../data/processed/events.csv"

if not os.path.exists(domestic_events_path):
    print("Domestic events.csv not found — run 02_league_analytics.ipynb first "
          "to enable this cross-check.")
    domestic_international_matches = pd.DataFrame()

else:
    domestic_events = pd.read_csv(domestic_events_path)
    domestic_players = domestic_events[["Player", "Team", "Competition"]].drop_duplicates()

    def normalize_name(name):
        # "Joan Ajao" -> frozenset({"joan", "ajao"})
        # "AJAO Joan" -> frozenset({"ajao", "joan"})
        # Same set either way, regardless of which name format was used.
        tokens = str(name).strip().split()
        return frozenset(t.lower() for t in tokens if t)

    domestic_players = domestic_players.copy()
    domestic_players["NameKey"] = domestic_players["Player"].apply(normalize_name)

    # Only match on names with 2+ tokens on both sides, to avoid a bare
    # single first name (e.g. "John") matching everyone else named John.
    domestic_players = domestic_players[domestic_players["NameKey"].apply(len) >= 2]

    if len(all_intl_matches) == 0 and "intl_events_df" not in dir():
        print("No international events available yet — run the scraping cells above first.")
        domestic_international_matches = pd.DataFrame()
    else:
        # Restrict to Kenya\'s own players internationally — matching a
        # domestic Kenyan league player against an opponent nation\'s squad
        # would be meaningless noise, not a real pathway to check.
        intl_players = intl_events_df[intl_events_df["Team"] == "Kenya"][
            ["Player", "Team", "CompetitionKey"]
        ].drop_duplicates()

        intl_players = intl_players.copy()
        intl_players["NameKey"] = intl_players["Player"].apply(normalize_name)
        intl_players = intl_players[intl_players["NameKey"].apply(len) >= 2]

        matches = domestic_players.merge(
            intl_players,
            on="NameKey",
            suffixes=("_Domestic", "_International")
        )

        domestic_international_matches = matches[[
            "Player_Domestic", "Team_Domestic", "Competition",
            "Player_International", "CompetitionKey"
        ]].rename(columns={
            "Player_Domestic": "DomesticName",
            "Team_Domestic": "DomesticClub",
            "Competition": "DomesticCompetition",
            "Player_International": "InternationalName",
            "CompetitionKey": "InternationalCompetition",
        }).drop_duplicates()

        print(f"Found {len(domestic_international_matches)} possible name match(es) "
              f"between domestic and international records.")
        print()
        print("⚠ IMPORTANT: these are NAME matches only, not confirmed identity")
        print("matches. Two different people can share a name. Verify manually")
        print("before citing any specific match.")

domestic_international_matches


Found 16 possible name match(es) between domestic and international records.

⚠ IMPORTANT: these are NAME matches only, not confirmed identity
matches. Two different people can share a name. Verify manually
before citing any specific match.


,DomesticName,DomesticClub,DomesticCompetition,InternationalName,InternationalCompetition
0,Joan Anjao,Blazers Hockey Club,PLW,ANJAO Joan,AFCON-W-2025
1,Vivian Onyango,Blazers Hockey Club,PLW,ONYANGO Vivian,AFCON-W-2022
2,Faith Amondi,Lakers Hockey Club,PLW,AMONDI Faith,JAC-W-2024
3,Grace Bwire,Strathmore University Ladies,PLW,BWIRE Grace,AFCON-W-2022
4,Grace Bwire,Strathmore University Ladies,PLW,BWIRE Grace,AFCON-W-2025
5,Naom Kemunto,USIU – A Ladies,PLW,KEMUNTO Naom,AFCON-W-2022
6,Naom Kemunto,USIU – A Ladies,PLW,KEMUNTO Naom,AFCON-W-2025
7,Robert Masibo,Kenya Police,PLM,MASIBO Robert,AFCON-M-2017
8,Festus Onyango,Warriors,PLM,ONYANGO Festus,AFCON-M-2017
9,Festus Onyango,Warriors,PLM,ONYANGO Festus,AFCON-M-2022


In [19]:
# Cell 14 - Save Linkage Results

if len(domestic_international_matches) > 0:
    domestic_international_matches.to_csv(
        "../data/processed/domestic_international_links.csv",
        index=False
    )
    print("Saved domestic_international_links.csv")
else:
    print("No matches to save this run.")


Saved domestic_international_links.csv


## Africa Cup for Club Championships — Medal History

A separate data source from everything above: national-team competitions
live on FIH\'s Tournament Management System (`tms.fih.ch`), but the
Africa Cup for Club Championships (ACCC) — where Kenyan *clubs*, not
the national team, compete — is reported on the African Hockey
Federation\'s own site (`africahockey.org`), confirmed by direct
inspection.

**Scope, stated honestly:** this captures the **medal-position summary
table** (year, host city, gold/silver/bronze) from 1988 through 2026,
which is real structured HTML — confirmed before writing this scraper.
It does **not** capture match-by-match goals or cards for this
competition. That detail exists only in individual prose news articles
(e.g. *"BRONZE MEDAL (M): PORT FOUAD (4-3) SIKH UNION SCORES: ..."*),
with no guaranteed consistent format between articles or years — a
genuinely different, harder scraping problem than anything else in this
notebook, and one that hasn\'t been attempted here. This is a documented
limitation, not an oversight.

Even at this level, the data is already worth having: it directly
confirms **Lakers Hockey Club of Kenya won the 2026 Women\'s ACCC title**,
among many other Kenyan club medals across the competition\'s history
(Simba Union, Kenya Armed Forces, Post & Telecomm, Sliders Club, Telkom
Kenya).


In [20]:
# Cell 15 - Scrape ACCC Medal History

# Confirmed via direct diagnostic against the live page: this table uses
# a MULTI-ROW header (e.g. a merged "Results" heading spanning row 0,
# then the real "Year/City/1/2/3" labels on row 1) — pandas.read_html()
# doesn\'t recognize merged headers, so it falls back to numeric column
# names [0,1,2,3,4] and pushes BOTH header rows down as if they were
# data. The fix: search the first few ROWS (not the column names) for
# identifying text, locate where real data actually starts, and read
# values by position from there.

ACCC_URL = "https://www.africahockey.org/africa-cup-for-club-championships/"

driver.get(ACCC_URL)

try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
except TimeoutException:
    print(f"⚠ Page did not load in time: {ACCC_URL}")

time.sleep(2)

html = driver.page_source

try:
    tables = pd.read_html(io.StringIO(html), flavor="lxml")
except (ValueError, ImportError):
    tables = []

print(f"Found {len(tables)} table(s) on the page.")

accc_medals = []

def row_text(row):
    return " ".join(str(v).strip().lower() for v in row.tolist())

for t in tables:

    header_row_idx = None
    header_type = None

    for row_idx in range(min(3, len(t))):
        text = row_text(t.iloc[row_idx])
        if "year" in text and "city" in text:
            header_row_idx = row_idx
            header_type = "historical"
            break
        elif "event name" in text:
            header_row_idx = row_idx
            header_type = "recent"
            break

    if header_row_idx is None:
        continue  # not a results table (e.g. navigation or unrelated table on the page)

    if header_type == "historical":
        # Columns by position: Edition, City, Gold, Silver, Bronze
        for i in range(header_row_idx + 1, len(t)):
            row = t.iloc[i]
            edition = str(row.iloc[0]).strip()
            if not edition or edition.lower() == "nan":
                continue
            accc_medals.append({
                "Edition": row.iloc[0],
                "City": row.iloc[1] if len(row) > 1 else None,
                "Gold": row.iloc[2] if len(row) > 2 else None,
                "Silver": row.iloc[3] if len(row) > 3 else None,
                "Bronze": row.iloc[4] if len(row) > 4 else None,
                "Category": "Historical",
            })

    elif header_type == "recent":
        # Row immediately after the header is a Male/Female sub-header
        # for the merged "Winners" column — skip that row too, real
        # data starts two rows down. Columns by position: Edition (0),
        # Dates (1), Venue (2), City/Country (3), Male winner (4),
        # Female winner (5).
        for i in range(header_row_idx + 2, len(t)):
            row = t.iloc[i]
            edition = str(row.iloc[0]).strip()
            if not edition or edition.lower() == "nan":
                continue
            accc_medals.append({
                "Edition": row.iloc[0],
                "City": row.iloc[3] if len(row) > 3 else None,
                "Gold": row.iloc[4] if len(row) > 4 else None,
                "Silver": row.iloc[5] if len(row) > 5 else None,
                "Bronze": None,
                "Category": "Recent",
            })

accc_medals_df = pd.DataFrame(accc_medals)

kenya_marker = accc_medals_df.astype(str).apply(
    lambda row: row.str.contains("Kenya", case=False, na=False).any(), axis=1
) if len(accc_medals_df) > 0 else pd.Series(dtype=bool)

if len(accc_medals_df) > 0:
    accc_medals_df["KenyaInvolved"] = kenya_marker

print(f"\nParsed {len(accc_medals_df)} edition(s), "
      f"{int(kenya_marker.sum()) if len(accc_medals_df) > 0 else 0} with Kenyan club involvement mentioned.")

accc_medals_df


Found 2 table(s) on the page.

Parsed 34 edition(s), 14 with Kenyan club involvement mentioned.


,Edition,City,Gold,Silver,Bronze,Category,KenyaInvolved
0,ACCC 1988,Cairo (Men),Sharkia Club (Egypt),Bendel Flickers Club (Nigeria),Police Union Club (Egypt),Historical,False
1,ACCC 1988,Blantyre (Men),Sharkia Club (Egypt),Zamalek Club (Egypt),Union Bank Club (Nigeria),Historical,False
2,ACCC 1990,Casablanca (Men),Sharkia Club (Egypt),Simba Union Club (Kenya),Golden Sticks (Ghana),Historical,True
3,ACCC 1991,Bulawayo (Men),Sharkia Club (Egypt),Exchequers Club (Ghana),Queens Club (Zimbabwe),Historical,False
4,ACCC 1992,Nairobi (Men),Sharkia Club (Egypt),Simba Union Club (Kenya),Trustee Club (Ghana),Historical,True
5,ACCC 1993,Cairo (Men),Sharkia Club (Egypt),Jeppe Quondam (RSA),Police Union Club (Egypt),Historical,False
6,ACCC 1994,Blantyre (Men),Sharkia Club (Egypt),Technikon Natal (RSA),Steel & Iron Club (Egypt),Historical,False
7,ACCC 1995,Accra (Men),Sharkia Club (Egypt),Yobe Desert Rollers (Nigeria),Police Union Club (Egypt),Historical,False
8,ACCC 1996,Bulawayo (Men),Sharkia Club (Egypt),Steel & Iron Club (Egypt),Supa Sweet Wits (Egypt),Historical,False
9,ACCC 1996,Bulawayo (Women),Old Hararians (Zimbabwe),Ramblers Club (Namibia),Bulawayo Athletic Club (Zimbabwe),Historical,False


In [21]:
# Cell 16 - Save ACCC Medal History

if len(accc_medals_df) > 0:
    accc_medals_df.to_csv("../data/raw/accc_medal_history.csv", index=False)
    print("Saved accc_medal_history.csv")
else:
    print("No ACCC data parsed — check the page structure hasn\'t changed, "
          "or inspect the raw tables list above manually.")


Saved accc_medal_history.csv


In [22]:
# DIAGNOSTIC - ACCC Table Structure (run this, then paste me the full output)

driver.get(ACCC_URL)
try:
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
except TimeoutException:
    pass
time.sleep(2)

html = driver.page_source
tables = pd.read_html(io.StringIO(html), flavor="lxml")

print(f"Found {len(tables)} table(s)\n")
for idx, t in enumerate(tables):
    print(f"=== TABLE {idx} ===")
    print("Columns:", list(t.columns))
    print(t.head(3))
    print()


Found 2 table(s)

=== TABLE 0 ===
Columns: [0, 1, 2, 3, 4]
           0            1                     2  \
0        NaN          NaN               Results   
1       Year         City                     1   
2  ACCC 1988  Cairo (Men)  Sharkia Club (Egypt)   

                                3                          4  
0                         Results                    Results  
1                               2                          3  
2  Bendel Flickers Club (Nigeria)  Police Union Club (Egypt)  

=== TABLE 1 ===
Columns: [0, 1, 2, 3, 4, 5]
            0      1      2             3        4        5
0  Event Name  Dates  Venue  City/Country  Winners  Winners
1         NaN    NaN    NaN           NaN     Male   Female
2   ACCC 2010    NaN    NaN           NaN      NaN      NaN



In [23]:
# DIAGNOSTIC - International Match Page Structure
# (run this on the FIRST Kenya match URL found, then paste me the full output)

test_url = list(kenya_match_links.values())[0][0] if any(kenya_match_links.values()) else None

if test_url is None:
    print("No match URLs discovered — run the discovery cell above first.")
else:
    print("Testing:", test_url)

    driver.get(test_url)
    try:
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    except TimeoutException:
        pass
    time.sleep(2)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text("\n", strip=True)

    print("=== First 600 characters of page text ===")
    print(page_text[:600])
    print()

    print("=== Headings found (h1/h2/h3) ===")
    for h in soup.find_all(["h1","h2","h3"]):
        t = h.get_text(strip=True)
        if t:
            print(repr(t))
    print()

    try:
        tables = pd.read_html(io.StringIO(html), flavor="lxml")
        print(f"=== {len(tables)} table(s) found ===")
        for idx, t in enumerate(tables):
            print(f"\nTable {idx} columns:", list(t.columns))
            print(t.head(3))
    except (ValueError, ImportError) as e:
        print("pd.read_html found no tables:", e)


Testing: https://tms.fih.ch/matches/10697


=== First 600 characters of page text ===
International Hockey Federation: Altiusrt
Login
Help
0
/ 1
What are the statistics terms' definitions?
Help Centre
Search
Home
2017 Africa Cup of Nations (M)
GHA v KEN
Help for this page (1)
What are the statistics terms' definitions?
Mark all as Read
Help Centre
2017 Africa Cup of Nations (M)
RR
Ghana
3 - 0
Official
Kenya
1st Quarter
Ghana
Kenya
15'
2nd Quarter
Ghana
Kenya
30'
3rd Quarter
Ghana
Kenya
45'
4th Quarter
Ghana
Kenya
60'
Lineups
Goals
Cards
Officials
Head to Head
Details
Awards
Ghana
#
Name
Min
1st
2nd
3rd
4th
1
KARIKARI Jeffrey (GK)
2
NSALBINI Salya (C)
X
4
ABBIW Charles
X
5
ASAMO

=== Headings found (h1/h2/h3) ===
'2017 Africa Cup of Nations (M)'
'RR'
'Ghana'
'3 - 0'
'Kenya'
'Goals'
'Card Detail'
'Match Officials'
'Head to Head Matches'
'Match Details'
'Match Awards'

=== 10 table(s) found ===

Table 0 columns: [0]
                                             0
0  What are the statistics terms' definitions?
1                      

## Advanced Match Statistics (Possession, Shots, Circle Entries, PC/PS)

FIH publishes a separate per-match statistics report — confirmed via
direct inspection to be a **PDF document** (`{match_url}/reports/statistics`,
served with `mime_type: application/pdf`), not an HTML page like
everything else scraped in this project. It contains genuinely
different data from the Goals/Cards tables already captured: Possession
%, Shots, Shots on Target, Circle Entries, and Penalty Corners/Strokes,
broken down by quarter.

**Confirmed risk, checked before building this:** every real example
found during reconnaissance — four different matches, different
competitions — showed **every single value as zero**. That\'s the
signature of an optional report that match officials frequently don\'t
fill in, the same pattern already confirmed for the domestic KHU goal-
event data. This module reports a coverage check **first**, before any
analysis leans on this data, exactly for that reason.

**Testing limitation, stated directly:** every other scraper in this
notebook was tested against a reconstruction of real diagnostic output
before being handed to you. This one could not be — downloading and
parsing an actual PDF file requires network access this environment
does not have. The code below is a genuine, careful attempt (using
`pdfplumber`, which reads PDF layout/position rather than guessing at
scrambled plain text), but **its first real test is your next run**,
not mine. Treat the coverage check output as the actual verification.


In [24]:
# Cell 17 - Fetch Match Statistics PDFs

import requests
import pdfplumber
import io as io_module

STAT_CATEGORIES = ["Penalty Corners", "Circle Entries", "Shots", "Shots on Target", "Possession"]

def fetch_and_parse_statistics_pdf(match_url):
    """
    Downloads the per-match statistics PDF and extracts whatever numbers
    it can find near each known stat category label. Returns None if the
    PDF can\'t be fetched or parsed at all (not the same as "found zeros" —
    that distinction matters for the coverage check below).
    """
    stats_url = match_url.rstrip("/") + "/reports/statistics"

    try:
        resp = requests.get(stats_url, timeout=20)
        if resp.status_code != 200 or not resp.content:
            return None
    except requests.RequestException:
        return None

    try:
        with pdfplumber.open(io_module.BytesIO(resp.content)) as pdf:
            full_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
    except Exception:
        return None

    if not full_text.strip():
        return None

    result = {}
    for category in STAT_CATEGORIES:
        idx = full_text.find(category)
        if idx == -1:
            result[category] = None
            continue
        # Grab the text between this category label and the next one (or
        # end of text), and pull out every number in that chunk — exact
        # per-team/per-quarter attribution isn\'t reliable from scrambled
        # PDF text layout, but "were there any non-zero numbers here at
        # all" is a robust, honest signal regardless of column order.
        next_positions = [full_text.find(c, idx + 1) for c in STAT_CATEGORIES if full_text.find(c, idx + 1) != -1]
        # The last stat category has no following category to bound it,
        # which (confirmed against a real PDF\'s extracted text) let the
        # chunk run all the way into the document\'s trailing boilerplate
        # (page numbers, copyright year) and get misread as real data.
        # Bound every chunk to a fixed max length instead, generous enough
        # for the 10 data values + 2 team codes this section actually
        # contains, but short enough to exclude the footer.
        MAX_CHUNK_LENGTH = 80
        boundary_candidates = next_positions + [idx + MAX_CHUNK_LENGTH]
        end = min(boundary_candidates)
        end = min(end, len(full_text))
        chunk = full_text[idx:end]
        numbers = [int(n) for n in re.findall(r"\b\d+\b", chunk)]
        result[category] = numbers

    return result


print("fetch_and_parse_statistics_pdf() ready.")


fetch_and_parse_statistics_pdf() ready.


## Module: FIH World Rankings — Kenya's Position Over Time, and Among African Nations

Two confirmed, real, distinct sources on FIH's own website:

1. Kenya's own ranking page (`fih.hockey/outdoor-rankings/kenya-men-hockey-rankings-41`) \u2014 confirmed directly by fetching it: current rank 49, 1446.21 points, with a working year filter (2021\u20132025) and match-by-match rank history.
2. The main outdoor rankings page (`fih.hockey/outdoor-hockey-rankings`) \u2014 confirmed directly by fetching it: the full list of every ranked nation, with confederation (e.g. "African Hockey Federation") shown per team, letting Kenya\'s position among African nations be computed honestly from real data rather than assumed to exist as a separately-published official ranking.

**Confirmed working: Kenya Men.** Kenya Women\'s individual ranking page URL was not confirmed in this session (the numeric ID isn\'t predictable from the pattern) \u2014 this follows the same "build what\'s confirmed, note what needs the same confirmation later" approach used for the Masters Hockey competitions earlier in this project.


In [25]:
# Cell R1 - Kenya World Ranking Scraper (Men)

KENYA_RANKING_URL = "https://www.fih.hockey/outdoor-rankings/kenya-men-hockey-rankings-41"

def scrape_kenya_world_ranking(url):
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        raise RuntimeError(f"Page did not load in time: {url}")
    time.sleep(3)

    page_text = BeautifulSoup(driver.page_source, "html.parser").get_text(" ", strip=True)

    # Current rank and points — confirmed real format from direct
    # inspection: "KENYA Rank 49 Current Points 1446.21"
    current_match = re.search(r"Rank\s+(\d+)\s+Current Points\s+([\d.]+)", page_text)
    current_rank = int(current_match.group(1)) if current_match else None
    current_points = float(current_match.group(2)) if current_match else None

    # Match-by-match rank history — confirmed real format directly
    # tested against real fetched page text: each row reads
    # "{day} {month} {opponent} Continental Championship {points before}
    # {points exchanged} {rank before} {rank after} Details". The
    # earlier version of this regex assumed a "Rank Before)" label
    # preceded each row's numbers, matching zero real rows when tested
    # \u2014 fixed to match the row structure actually observed.
    history_pattern = re.compile(
        r"(\d{1,2}\s\w{3})\s+([A-Za-z][A-Za-z &]*?)\s+Continental Championship\s+"
        r"([\d.]+)\s+(-?[\d.]+)\s+(\d+)\s+(\d+)\s+Details"
    )
    history_rows = history_pattern.findall(page_text)

    history = [
        {"DateLabel": d, "Opponent": opp.strip(), "PointsBefore": float(pb), "PointsExchanged": float(pe), "RankBefore": int(rb), "RankAfter": int(ra)}
        for d, opp, pb, pe, rb, ra in history_rows
    ]

    return {
        "CurrentRank": current_rank,
        "CurrentPoints": current_points,
        "History": history,
    }


print("scrape_kenya_world_ranking() ready.")


scrape_kenya_world_ranking() ready.


In [26]:
# Cell R2 - Run Kenya Ranking Scraper

kenya_ranking_result = scrape_kenya_world_ranking(KENYA_RANKING_URL)
print(f"Kenya Men current rank: {kenya_ranking_result['CurrentRank']}")
print(f"Kenya Men current points: {kenya_ranking_result['CurrentPoints']}")
print(f"Rank-change history entries found: {len(kenya_ranking_result['History'])}")

kenya_ranking_df = pd.DataFrame([{
    "Gender": "Men",
    "CurrentRank": kenya_ranking_result["CurrentRank"],
    "CurrentPoints": kenya_ranking_result["CurrentPoints"],
}])

kenya_ranking_history_df = pd.DataFrame(kenya_ranking_result["History"])
if len(kenya_ranking_history_df) > 0:
    kenya_ranking_history_df["Gender"] = "Men"

kenya_ranking_history_df


Kenya Men current rank: 49
Kenya Men current points: 1445.77
Rank-change history entries found: 6


,DateLabel,Opponent,PointsBefore,PointsExchanged,RankBefore,RankAfter,Gender
0,18 Oct,Nigeria,1396.59,-50.81,50,64,Men
1,17 Oct,Zambia,1347.67,48.91,64,50,Men
2,15 Oct,Nigeria,1264.91,82.76,79,65,Men
3,14 Oct,Ghana,1306.00,-41.08,71,79,Men
4,12 Oct,South Africa,1306.00,0.00,71,71,Men
5,11 Oct,Egypt,1308.54,-2.54,71,71,Men


In [27]:
# Cell R3 - African Confederation Comparison (from the main rankings page)

MAIN_RANKINGS_URL = "https://www.fih.hockey/outdoor-hockey-rankings"

def scrape_african_rankings(url):
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        raise RuntimeError(f"Page did not load in time: {url}")
    time.sleep(3)

    page_text = BeautifulSoup(driver.page_source, "html.parser").get_text(" ", strip=True)

    # Confirmed real row format from direct inspection: e.g.
    # "49 KENYA African Hockey Federation Oct 18, 2025 1446.21"
    row_pattern = re.compile(
        r"(\d{1,3})\s+([A-Z][A-Z &.]+?)\s+(African Hockey Federation)\s+"
        r"([A-Za-z]{3}\s\d{1,2},\s\d{4}|-)\s+([\d.]+)"
    )

    rows = [
        {"Rank": int(m.group(1)), "Team": m.group(2).strip().title(), "LastMatch": m.group(4), "Points": float(m.group(5))}
        for m in row_pattern.finditer(page_text)
    ]

    return pd.DataFrame(rows)


african_rankings_df = scrape_african_rankings(MAIN_RANKINGS_URL)
african_rankings_df["Gender"] = "Men"
print(f"African Hockey Federation nations found: {len(african_rankings_df)}")
african_rankings_df.sort_values("Rank")


African Hockey Federation nations found: 12


,Rank,Team,LastMatch,Points,Gender
0,13,South Africa,"Aug 28, 2026",2531.46,Men
1,16,Egypt,"Mar 07, 2026",2308.99,Men
2,33,Ghana,"Oct 17, 2025",1722.31,Men
3,35,Nigeria,"Oct 18, 2025",1700.42,Men
4,49,Kenya,"Oct 18, 2025",1445.77,Men
5,70,Namibia,"Aug 21, 2024",1361.08,Men
6,88,Uganda,"Nov 05, 2023",1236.94,Men
7,89,Zimbabwe,"Aug 21, 2024",1236.73,Men
8,90,Zambia,"Oct 17, 2025",1214.09,Men
9,96,Malawi,"Sep 03, 2022",753.92,Men


### Kenya Women — via the Women's toggle on the main rankings page

Rather than needing Kenya Women's specific individual ranking page URL
(not confirmed in this session \u2014 the numeric ID isn't predictable from
the pattern), this uses the same confirmed-working main rankings page
and clicks its "Women's" toggle directly, since that page has already
been confirmed to genuinely list every nation with confederation and
points.

**Honest note: this specific click interaction has not been tested
against the live site** (unlike the regex patterns used elsewhere in
this notebook, which were verified against real fetched text). If it
fails, the diagnostic output below will show exactly what went wrong
rather than silently returning nothing \u2014 paste that output back for
a real fix, the same way every other real bug in this project has been
resolved.


In [28]:
# Cell R5 - Kenya Women, via the Women\'s Toggle (Untested Interaction - Check Output Carefully)

def scrape_african_rankings_women(url):
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        raise RuntimeError(f"Page did not load in time: {url}")
    time.sleep(3)

    # Try a few reasonable ways to find and click a "Women\'s" toggle,
    # since the exact element structure wasn\'t directly inspectable
    # without a live browser session. Reports clearly which strategy
    # worked, or that none did, rather than failing silently.
    clicked = False
    strategies = [
        (By.PARTIAL_LINK_TEXT, "Women"),
        (By.XPATH, "//*[contains(text(), \"Women's\")]"),
        (By.XPATH, "//button[contains(., 'Women')]"),
    ]
    for by, selector in strategies:
        try:
            el = driver.find_element(by, selector)
            el.click()
            clicked = True
            print(f"Clicked Women's toggle using strategy: {by} = {selector!r}")
            break
        except Exception:
            continue

    if not clicked:
        print("Could not find/click a Women's toggle with any of the tried strategies.")
        print("The page text below is whatever loaded by default (likely still Men's) \u2014")
        print("inspect it to find the right element, then this cell can be corrected.")

    time.sleep(2)
    page_text = BeautifulSoup(driver.page_source, "html.parser").get_text(" ", strip=True)

    row_pattern = re.compile(
        r"(\d{1,3})\s+([A-Z][A-Z &.]+?)\s+(African Hockey Federation)\s+"
        r"([A-Za-z]{3}\s\d{1,2},\s\d{4}|-)\s+([\d.]+)"
    )
    rows = [
        {"Rank": int(m.group(1)), "Team": m.group(2).strip().title(), "LastMatch": m.group(4), "Points": float(m.group(5))}
        for m in row_pattern.finditer(page_text)
    ]

    return pd.DataFrame(rows), clicked


african_rankings_women_df, women_toggle_clicked = scrape_african_rankings_women(MAIN_RANKINGS_URL)
print(f"\nWomen\'s toggle click succeeded: {women_toggle_clicked}")
print(f"African nations found: {len(african_rankings_women_df)}")
if len(african_rankings_women_df) > 0:
    african_rankings_women_df["Gender"] = "Women"
    kenya_women_row = african_rankings_women_df[african_rankings_women_df["Team"] == "Kenya"]
    if len(kenya_women_row) > 0:
        kw_rank = int(kenya_women_row.iloc[0]["Rank"])
        kw_points = kenya_women_row.iloc[0]["Points"]
        print(f"Kenya Women found: Rank {kw_rank}, {kw_points} points")
    else:
        print("Kenya not found in the extracted rows - the toggle likely did not switch to Women's data. Check women_toggle_clicked above.")
african_rankings_women_df.sort_values("Rank") if len(african_rankings_women_df) > 0 else african_rankings_women_df


Clicked Women's toggle using strategy: xpath = "//button[contains(., 'Women')]"



Women's toggle click succeeded: True
African nations found: 12
Kenya Women found: Rank 32, 1573.85 points


,Rank,Team,LastMatch,Points,Gender
0,18,South Africa,"Aug 27, 2026",2050.93,Women
1,32,Kenya,"Oct 18, 2025",1573.85,Women
2,33,Ghana,"Oct 18, 2025",1570.90,Women
3,39,Nigeria,"Oct 18, 2025",1311.22,Women
4,52,Namibia,"Jul 24, 2026",1190.39,Women
5,61,Zimbabwe,"Aug 21, 2024",1137.77,Women
6,66,Egypt,"Oct 15, 2025",1111.75,Women
7,76,Zambia,"Aug 21, 2024",960.64,Women
8,78,Uganda,"Jan 23, 2022",870.43,Women
9,79,Malawi,"Sep 03, 2022",731.63,Women


In [29]:
# Cell R4 - Save World Ranking Data

kenya_ranking_df.to_csv("../data/processed/kenya_world_ranking.csv", index=False)
kenya_ranking_history_df.to_csv("../data/processed/kenya_ranking_history.csv", index=False)
african_rankings_df.to_csv("../data/processed/african_rankings.csv", index=False)

print("Saved kenya_world_ranking.csv, kenya_ranking_history.csv, and african_rankings.csv")


Saved kenya_world_ranking.csv, kenya_ranking_history.csv, and african_rankings.csv


In [30]:
# Cell R6 - Save Women\'s Data (Only If Genuinely Found)

if len(african_rankings_women_df) > 0 and women_toggle_clicked:
    kenya_women_check = african_rankings_women_df[african_rankings_women_df["Team"] == "Kenya"]
    if len(kenya_women_check) > 0:
        women_ranking_df = pd.DataFrame([{
            "Gender": "Women",
            "CurrentRank": int(kenya_women_check.iloc[0]["Rank"]),
            "CurrentPoints": kenya_women_check.iloc[0]["Points"],
        }])
        combined_ranking_df = pd.concat([kenya_ranking_df, women_ranking_df], ignore_index=True)
        combined_ranking_df.to_csv("../data/processed/kenya_world_ranking.csv", index=False)

        combined_african_df = pd.concat([african_rankings_df, african_rankings_women_df], ignore_index=True)
        combined_african_df.to_csv("../data/processed/african_rankings.csv", index=False)
        print("Saved combined Men\'s + Women\'s ranking data.")
    else:
        print("Kenya Women\'s data not confirmed in the scraped rows \u2014 keeping Men\'s-only data as saved by Cell R4.")
else:
    print("Women\'s toggle did not produce usable data \u2014 keeping Men\'s-only data as saved by Cell R4.")


Saved combined Men's + Women's ranking data.


## Module: FIH World Cup 2026 Standings — Moved to `05_worldcup_live.ipynb`

**Audit note (fixed):** this notebook used to carry its own copy of the World Cup pool scraper. That copy predates the click-targeting fix described in the project chat log (Kenya Hockey Union chat export) — it searched for *any* element containing the tab's text, which matched invisible `<title>`/`<meta>` tags and the schedule widget's `<p class="pool venue">` labels before ever reaching the real, visible Pool/Gender tab. Confirmed directly in this notebook's own last recorded run: every pool click failed (`could not click this pool's tab, skipping`) and it saved an **empty** `world_cup_standings.csv`.

**The real, fixed scraper now lives only in `05_worldcup_live.ipynb`** (it uses `find_real_tab()`, which filters out non-interactive tags/classes and prefers a real button/link/tab-role element — confirmed working: 32 rows, 8 real pool combinations, correct Men/Women splits).

**Why the old cells were removed rather than left in place:** both notebooks wrote to the same file, `../data/processed/world_cup_standings.csv`. If a scheduled run executed this notebook (`04`) after `05` had already produced good data — e.g. the weekly domestic pipeline running `01→04` on a schedule that overlaps with `05`'s more frequent World Cup refresh — the dead code here would have silently overwritten good live data with an empty table. Removing it here (instead of just leaving it broken) closes that race condition permanently. If `04` needs World Cup context again in the future, import the already-fixed functions from `05` rather than re-forking a second copy.


In [31]:
# Cell 18 - Fetch Statistics for All Kenya Matches, With Coverage Check FIRST

# Reuses the match URLs already discovered above — no new match
# discovery needed, just a second data point per match already found.

kenya_match_urls = [m["url"] for m in all_intl_matches] if len(all_intl_matches) > 0 else []

stats_results = {}   # url -> parsed stats dict, or None if unavailable

for i, url in enumerate(kenya_match_urls, start=1):
    print(f"[{i}/{len(kenya_match_urls)}] {url}")
    parsed = fetch_and_parse_statistics_pdf(url)
    stats_results[url] = parsed
    if parsed is None:
        print("  \u26a0 Could not fetch or parse this statistics PDF.")
    else:
        any_nonzero = any(any(n > 0 for n in nums) for nums in parsed.values() if nums)
        if any_nonzero:
            print("  \u2713 Contains non-zero data")
        else:
            print("  \u26a0 All values are zero (or unfilled)")


# --- Coverage check FIRST, reported PER CATEGORY — a real match already
# confirmed during testing that these categories are NOT all-or-nothing:
# Penalty Corners can be genuinely populated while Circle Entries, Shots,
# and Possession are simultaneously all zero on the very same match. A
# single blended "any stat present" number would hide that pattern.
attempted = len(kenya_match_urls)
fetched_ok = sum(1 for v in stats_results.values() if v is not None)

print(f"\n{chr(61)*70}")
print("ADVANCED STATISTICS COVERAGE CHECK (per category)")
print(f"{chr(61)*70}")
print(f"Matches attempted        : {attempted}")
print(f"PDF fetched successfully : {fetched_ok}")

if fetched_ok > 0:
    print()
    for category in STAT_CATEGORIES:
        with_data = sum(
            1 for v in stats_results.values()
            if v is not None and v.get(category) and any(n > 0 for n in v[category])
        )
        rate = (with_data / fetched_ok * 100) if fetched_ok > 0 else 0
        print(f"  {category:20s}: {with_data}/{fetched_ok} matches ({rate:.1f}%) have non-zero data")

    print("\nUse these per-category rates, not a single blended number, to decide which")
    print("of these stats are trustworthy enough to analyze further and which are too")
    print("sparsely populated to draw conclusions from.")


[1/30] https://tms.fih.ch/matches/10697


  ⚠ All values are zero (or unfilled)
[2/30] https://tms.fih.ch/matches/10700


  ⚠ All values are zero (or unfilled)
[3/30] https://tms.fih.ch/matches/10702


  ⚠ All values are zero (or unfilled)
[4/30] https://tms.fih.ch/matches/10708


  ⚠ All values are zero (or unfilled)
[5/30] https://tms.fih.ch/matches/10712


...

  ✓ Contains non-zero data
[27/30] https://tms.fih.ch/matches/20972


  ✓ Contains non-zero data
[28/30] https://tms.fih.ch/matches/20974


  ✓ Contains non-zero data
[29/30] https://tms.fih.ch/matches/20977


  ✓ Contains non-zero data
[30/30] https://tms.fih.ch/matches/20980


  ✓ Contains non-zero data

ADVANCED STATISTICS COVERAGE CHECK (per category)
Matches attempted        : 30
PDF fetched successfully : 30

  Penalty Corners     : 0/30 matches (0.0%) have non-zero data
  Circle Entries      : 17/30 matches (56.7%) have non-zero data
  Shots               : 0/30 matches (0.0%) have non-zero data
  Shots on Target     : 0/30 matches (0.0%) have non-zero data
  Possession          : 0/30 matches (0.0%) have non-zero data

Use these per-category rates, not a single blended number, to decide which
of these stats are trustworthy enough to analyze further and which are too
sparsely populated to draw conclusions from.
